# LeRover CNN Policy — Google Colab Training

### Before you run this notebook
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Upload your dataset to Google Drive** — create this folder structure:
   ```
   MyDrive/
   └── leorovervla/
       └── data/
           └── leorover_cnn_white/      ← drag-and-drop from your laptop
               ├── episodes/
               └── raw/
   ```
3. Edit the **Configuration** cell below, then run all cells in order.

Checkpoints are saved to `/content/runs/` during training and the best one
is copied to Drive + offered as a download at the end.

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone repository

In [ ]:
import os

REPO_DIR = "/content/leorovervla"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MichaelAguadze/leorovervla.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

## 3 · Install dependencies

In [ ]:
!pip install -q av pyarrow
print("Dependencies ready.")

## 4 · Configuration

**Edit this cell** to match your dataset and training preferences.

In [ ]:
# ── Edit these ─────────────────────────────────────────────────────────────
DATASET_NAME   = "leorover_cnn_white"   # folder name inside DRIVE_DATA_DIR
TAPE_COLOR     = "white"                # "red" | "white" | "blue" | "green" | None
EPOCHS         = 30
BATCH_SIZE     = 64                     # increase if GPU allows (T4: 64-128)
LEARNING_RATE  = 3e-4
RUN_NAME       = "white_v1"             # used for output folder name
DRIVE_DATA_DIR = "/content/drive/MyDrive/leorovervla/data"
# ───────────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, "/content/leorovervla")

EPISODES_DIR   = f"{DRIVE_DATA_DIR}/{DATASET_NAME}/episodes"
LOCAL_RUN_DIR  = f"/content/runs/{RUN_NAME}"          # fast I/O during training
DRIVE_RUN_DIR  = f"/content/drive/MyDrive/leorovervla/runs/{RUN_NAME}"  # permanent copy

print(f"Episodes dir  : {EPISODES_DIR}")
print(f"Run dir       : {LOCAL_RUN_DIR}")
print(f"Drive run dir : {DRIVE_RUN_DIR}")

## 5 · Verify dataset

In [ ]:
from pathlib import Path
from collections import Counter
from loop_cnn.dataset import discover_cnn_episodes

episodes = discover_cnn_episodes(Path(EPISODES_DIR), tape_color=TAPE_COLOR)

if not episodes:
    raise RuntimeError(
        f"No episodes found under {EPISODES_DIR}\n"
        "Check that your Drive folder is structured as described in Step 1."
    )

print(f"Found {len(episodes)} episodes")
for direction, count in sorted(Counter(e.direction for e in episodes).items()):
    print(f"  {direction}: {count}")

colors = Counter(e.tape_color for e in episodes)
print("Tape colors:", dict(colors))

## 6 · Train

In [ ]:
import shlex

tape_flag = f"--tape-color {TAPE_COLOR}" if TAPE_COLOR else ""

cmd = (
    f"python -m loop_cnn.train"
    f" --episodes-dir \"{EPISODES_DIR}\""
    f" --run-dir \"{LOCAL_RUN_DIR}\""
    f" --epochs {EPOCHS}"
    f" --batch-size {BATCH_SIZE}"
    f" --lr {LEARNING_RATE}"
    f" --num-workers 2"
    f" --device auto"
    f" {tape_flag}"
)

print("Running:", cmd)
!{cmd}

## 7 · Plot training curves

In [ ]:
import json
import glob
import math
import matplotlib.pyplot as plt

summary_files = sorted(glob.glob(f"{LOCAL_RUN_DIR}/*/training_summary.json"))
if not summary_files:
    print("No training_summary.json found — run the training cell first.")
else:
    with open(summary_files[-1]) as f:
        summary = json.load(f)

    history = summary["history"]
    epochs      = [r["epoch"]         for r in history]
    train_loss  = [r["train_loss"]    for r in history]
    val_loss    = [r["val_loss"]      for r in history]
    val_omega   = [r["val_mae_omega"] for r in history]
    val_vx      = [r["val_mae_vx"]    for r in history]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, train_loss, label="train")
    axes[0].plot(epochs, [v if not math.isnan(v) else None for v in val_loss], label="val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Huber loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, val_omega, label="mae_omega")
    axes[1].plot(epochs, val_vx,    label="mae_vx")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE (normalised)")
    axes[1].set_title("Val MAE")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f"Run: {RUN_NAME}  |  Best epoch: {summary['best_epoch']}  |  Best loss: {summary['best_metric']:.4f}")
    plt.tight_layout()

    curve_path = summary_files[-1].replace("training_summary.json", "training_curves.png")
    plt.savefig(curve_path, dpi=150)
    plt.show()
    print(f"Best epoch : {summary['best_epoch']}")
    print(f"Best loss  : {summary['best_metric']:.4f}")

## 8 · Save best checkpoint to Drive + download

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

best_candidates = sorted(Path(LOCAL_RUN_DIR).glob("*/checkpoints/best.pt"))
summary_candidates = sorted(Path(LOCAL_RUN_DIR).glob("*/training_summary.json"))

if not best_candidates:
    print("best.pt not found — training may not have completed.")
else:
    best_src     = best_candidates[-1]
    summary_src  = summary_candidates[-1] if summary_candidates else None
    run_subdir   = best_src.parent.parent.name   # e.g. run_20260528_143201

    drive_dest = Path(DRIVE_RUN_DIR) / run_subdir
    drive_dest.mkdir(parents=True, exist_ok=True)

    shutil.copy2(best_src, drive_dest / "best.pt")
    print(f"Copied best.pt → {drive_dest / 'best.pt'}")

    if summary_src:
        shutil.copy2(summary_src, drive_dest / "training_summary.json")
        print(f"Copied training_summary.json → {drive_dest}")

    # Also offer a direct browser download
    files.download(str(best_src))
    print("Download started for best.pt")